## [Locus: the paper](https://antfriend.github.io/locus_arxiv.pdf)
for this novel framework.   

The learnings:
## [Locus Logs](https://antfriend.github.io/index.html?ttdb=companion_arcprize.md&toot=lat-300lon10)

In [ ]:
# learn better
import os, subprocess

# Dump Kaggle/competition env vars for diagnostics
for k, v in sorted(os.environ.items()):
    if any(k.startswith(p) for p in ('KAGGLE', 'ARC', 'COMPETITION', 'GATEWAY')):
        print(f'[env] {k}={v}')

# Install competition wheels
_wdir = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
if not os.path.isdir(_wdir):
    _wdir = '/kaggle/input/datasets/danxray/companion-arc/wheels'
print(f'[wheels] {_wdir}')
import glob as _g
subprocess.run(['pip', 'install', '--no-deps', '-q'] + _g.glob(f'{_wdir}/*.whl'), check=True)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # -------------------------------------------------------------------
    # COMPETITION RERUN — use official ARC-AGI-3-Agents framework
    # with LOCUS route agent
    # -------------------------------------------------------------------
    print('[mode] COMPETITION RERUN — framework mode (locus agent)')

    # Write LOCUS route agent
    _locus_agent_src = [
        "import random as _random",
        "import numpy as _np",
        "from typing import Any",
        "from arcengine import FrameData, GameAction, GameState",
        "from agents.agent import Agent",
        "",
        "",
        "class LucusAgent(Agent):",
        '    """LOCUS-discovered routes for ARC-AGI-3 competition games.',
        "    Phase 1: adaptive route from first frame (ls20) or static known route.",
        "    Phase 2: systematic single-action sweeps (covers short repeat patterns).",
        "    Phase 3: random play.",
        '    """',
        "",
        "    MAX_ACTIONS = float('inf')",
        "",
        "    # Known routes for cd82 and sp80 (instance-stable).",
        "    # ls20 uses adaptive detection (_detect_l1_route) instead of a static route.",
        "    _ROUTES = {",
        '        "cd82": [3, 0, 1, 0, 0, 0, 1, 1, 1, 3, 2, 0, 4, 4, 2, 0, 0, 0, 1],',
        '        "sp80": [4, 3, 3, 3, 4, 2, 2, 1],',
        "    }",
        "",
        "    def __init__(self, *args: Any, **kwargs: Any) -> None:",
        "        super().__init__(*args, **kwargs)",
        '        prefix = self.game_id.split("-")[0] if "-" in self.game_id else self.game_id',
        "        self._route = list(self._ROUTES.get(prefix, []))",
        "        self._play_step = 0",
        "        self._prev_levels = 0",
        "        self._route_computed = False",
        "        self._systematic_queue = []",
        "        self._systematic_step = 0",
        "        self._simple = [a for a in GameAction if a.is_simple()]",
        "        self._rng = _random.Random(hash(self.game_id) & 0xFFFFFFFF)",
        '        print(f"[locus] {self.game_id}: route={len(self._route)} steps, actions={len(self._simple)}")',
        "",
        "    @staticmethod",
        "    def _detect_l1_route(grid):",
        '        """Compute L1 route: UP × n until block enters entity2 interior."""',
        "        try:",
        "            BLOCK, WALL, FLOOR = 12, 3, 5",
        "            positions = _np.argwhere(grid == BLOCK)",
        "            if not len(positions):",
        "                return None",
        "            block_row = int(positions[:, 0].min())",
        "            for r in range(2, 40):",
        "                walls = _np.where(grid[r] == WALL)[0]",
        "                if len(walls) < 2:",
        "                    continue",
        "                L, R = int(walls[0]), int(walls[-1])",
        "                if R - L < 4:",
        "                    continue",
        "                if _np.any(grid[r + 1, L + 1:R] == FLOOR):",
        "                    interior_top = r + 1",
        "                    ups = max(1, (block_row - interior_top) // 5)",
        "                    return [0] * ups",
        "        except Exception:",
        "            pass",
        "        return [0] * 7",
        "",
        "    def _get_grid(self, latest_frame, frames):",
        '        """Try to extract a 2D numpy grid from FrameData."""',
        "        for src in ([latest_frame] + list(frames or [])):",
        "            for attr in ('frame', 'obs', 'grid'):",
        "                val = getattr(src, attr, None)",
        "                if val is None:",
        "                    continue",
        "                try:",
        "                    arr = _np.array(val[0] if isinstance(val, (list, tuple)) else val)",
        "                    if arr.ndim == 2 and arr.shape[0] > 10:",
        "                        return arr",
        "                except Exception:",
        "                    pass",
        "        return None",
        "",
        "    def _build_systematic(self):",
        '        """Each action repeated 1..20 times — covers short repeat patterns exhaustively."""',
        "        n = len(self._simple)",
        "        seq = []",
        "        for repeat in range(1, 21):",
        "            for a in range(n):",
        "                seq.extend([a] * repeat)",
        "        return seq",
        "",
        "    def is_done(self, frames: list, latest_frame: FrameData) -> bool:",
        "        return latest_frame.state is GameState.WIN",
        "",
        "    def choose_action(self, frames: list, latest_frame: FrameData) -> GameAction:",
        "        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):",
        "            self._play_step = 0",
        "            self._prev_levels = 0",
        "            self._route_computed = False",
        "            self._systematic_queue = []",
        "            self._systematic_step = 0",
        "            return GameAction.RESET",
        "",
        "        # Adaptive route detection on first call",
        "        if not self._route_computed:",
        "            self._route_computed = True",
        '            if self.game_id.startswith("ls20"):',
        "                grid = self._get_grid(latest_frame, frames)",
        "                if grid is not None:",
        "                    detected = self._detect_l1_route(grid)",
        "                    if detected is not None:",
        "                        self._route = detected",
        '                        print(f"[locus] {self.game_id}: adaptive route {detected}")',
        "",
        "        # Reset route pointer when a new level is reached",
        "        current_levels = getattr(latest_frame, 'levels_completed', 0) or 0",
        "        if current_levels > self._prev_levels:",
        '            print(f"[locus] {self.game_id}: level {current_levels} — resetting route")',
        "            self._play_step = 0",
        "            self._route_computed = False",
        "            self._systematic_queue = []",
        "            self._systematic_step = 0",
        "            self._prev_levels = current_levels",
        "",
        "        n = max(len(self._simple), 1)",
        "",
        "        # Phase 1: known/adaptive route",
        "        if self._play_step < len(self._route):",
        "            idx = self._route[self._play_step] % n",
        "            action = self._simple[idx]",
        "            if action.is_simple():",
        '                action.reasoning = f"LOCUS route step {self._play_step}"',
        "            self._play_step += 1",
        "            return action",
        "",
        "        # Phase 2: systematic sweeps",
        "        if not self._systematic_queue:",
        "            self._systematic_queue = self._build_systematic()",
        "        if self._systematic_step < len(self._systematic_queue):",
        "            idx = self._systematic_queue[self._systematic_step] % n",
        "            action = self._simple[idx]",
        "            if action.is_simple():",
        '                action.reasoning = f"systematic {self._systematic_step}"',
        "            self._systematic_step += 1",
        "            return action",
        "",
        "        # Phase 3: random",
        "        action = self._rng.choice(self._simple) if self._simple else GameAction.RESET",
        "        if action.is_simple():",
        '            action.reasoning = "random fallback"',
        "        return action",
    ]
    with open('/kaggle/working/locus_agent.py', 'w') as _f:
        _f.write('\n'.join(_locus_agent_src) + '\n')

    # Wait for gateway to be ready
    subprocess.run([
        'curl', '--fail', '--retry', '999', '--retry-all-errors',
        '--retry-delay', '5', '--retry-max-time', '600',
        'http://gateway:8001/api/games'
    ], check=True)

    # Copy framework to writable location
    subprocess.run([
        'cp', '-r',
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents',
        '/kaggle/working/ARC-AGI-3-Agents'
    ], check=True)

    # Install locus agent into framework
    subprocess.run([
        'cp', '/kaggle/working/locus_agent.py',
        '/kaggle/working/ARC-AGI-3-Agents/agents/templates/locus_agent.py'
    ], check=True)

    # Override __init__.py — original has unmet deps that crash at import
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as _f:
        _f.write(
            'from typing import Type\n'
            'from dotenv import load_dotenv\n'
            'from .agent import Agent, Playback\n'
            'from .swarm import Swarm\n'
            'from .templates.random_agent import Random\n'
            'from .templates.locus_agent import LucusAgent\n'
            '\n'
            'load_dotenv()\n'
            '\n'
            'AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n'
            "    'random': Random,\n"
            "    'locus': LucusAgent,\n"
            '}\n'
        )

    # Write .env — gateway config
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as _f:
        _f.write(
            'SCHEME=http\n'
            'HOST=gateway\n'
            'PORT=8001\n'
            'ARC_API_KEY=locus_agent\n'
            'ARC_BASE_URL=http://gateway:8001/\n'
            'OPERATION_MODE=online\n'
            'ENVIRONMENTS_DIR=\n'
            'RECORDINGS_DIR=/kaggle/working/server_recording\n'
        )

    # Run LOCUS agent via framework
    _env = {**os.environ, 'MPLBACKEND': 'agg'}
    subprocess.run(
        ['python', 'main.py', '--agent', 'locus'],
        cwd='/kaggle/working/ARC-AGI-3-Agents',
        env=_env,
        check=False
    )

else:
    # -------------------------------------------------------------------
    # BATCH RUN — offline diagnostics (not the competition scoring path)
    # -------------------------------------------------------------------
    print('[mode] BATCH RUN — offline diagnostics')
    subprocess.run([
        'python3',
        '/kaggle/input/datasets/danxray/companion-arc/launch_competition.py'
    ], env=os.environ, check=False)
